Baseline Fraud Detection: Random Forest + SMOTE
This notebook establishes a baseline performance using traditional tabular features. We address class imbalance using the Synthetic Minority Over-sampling Technique (SMOTE).

Dataset: Credit Card Fraud Detection (Kaggle).

Dependencies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from imblearn.over_sampling import SMOTE

# Set plotting style
plt.style.use('ggplot')

Kaggle setup + download

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = "your_username"
os.environ['KAGGLE_KEY'] = "your_key"

!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip

Data Loading

In [ ]:
try:
    credit = pd.read_csv("creditcard.csv")
    print(f"Dataset loaded successfully with shape: {credit.shape}")
except FileNotFoundError:
    print("Error: creditcard.csv not found. Please download it from Kaggle.")

X = credit.drop("Class", axis=1)
y = credit["Class"]

Preprocessing & Balancing

In [ ]:
# Train-test split with stratification to maintain fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Apply SMOTE only to the training data to prevent data leakage
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

# Normalize features
scaler = StandardScaler()
X_train_bal = scaler.fit_transform(X_train_bal)
X_test = scaler.transform(X_test)

Model Training

In [ ]:
# Initialize and train the Baseline Random Forest
rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
rf.fit(X_train_bal, y_train_bal)

Evaluation

In [ ]:

pred = rf.predict_proba(X_test)[:,1]

print("Baseline Model: Random Forest + SMOTE")
print("AUROC:", roc_auc_score(y_test, pred))
print("AUPRC (Precision-Recall AUC):", average_precision_score(y_test, pred))
print(classification_report(y_test, rf.predict(X_test)))

Baseline Model: Random Forest + SMOTE
AUROC: 0.9854141783331228
AUPRC (Precision-Recall AUC): 0.8752540451107216
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.83      0.83        98

    accuracy                           1.00     56962
   macro avg       0.91      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962



## Final Results

- AUROC ≈ 0.985  
- AUPRC ≈ 0.875  
- F1-score ≈ 0.83  

This baseline model uses only tabular features.

## Limitation

The model treats transactions independently and does not capture relationships between accounts.

This motivates graph-based approaches like Node2Vec.